# YOLOv4-Tiny - Palm Oil FFB Ripeness Detection

**Inference and Visualization Notebook**

This notebook loads weights and artifacts from HuggingFace Hub (trained on Modal B200), then:
- Displays evaluation metrics for the 4 models
- Plots training loss and mAP comparisons
- Plots GA optimization progression
- Runs inference and visualizes bounding boxes on test images

No Darknet compilation required (uses OpenCV DNN module).

**Reference paper:** Salim and Suharjito (2023), Smart Agricultural Technology

## Step 1: Setup

In [ ]:
!pip install -q huggingface_hub roboflow opencv-python matplotlib pandas numpy tabulate

In [ ]:
HF_REPO = "dutaav/yolov4-tiny-hpo-ffb-maturity"
RUN_TAG = "h100-hankai"

ROBOFLOW_API_KEY = "YOUR_API_KEY"

## Step 2: Download artifacts from HuggingFace Hub

In [ ]:
import os, json
from pathlib import Path
from huggingface_hub import snapshot_download

local_dir = snapshot_download(
    repo_id=HF_REPO,
    repo_type="model",
    local_dir="/content/hf_repo",
    allow_patterns=[f"runs/{RUN_TAG}/*"],
)
REPO = Path(local_dir) / "runs" / RUN_TAG

print(f"REPO root: {REPO}")
for p in sorted(REPO.rglob("*")):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"  {p.relative_to(REPO)}  ({size_kb:,.1f} KB)")

In [ ]:
dataset_info = json.loads((REPO / "dataset_info.json").read_text())
metrics = json.loads((REPO / "artifacts/metrics.json").read_text())
ga_history = json.loads((REPO / "artifacts/ga_history.json").read_text())
training_info = json.loads((REPO / "artifacts/training_info.json").read_text())

NUM_CLASSES = dataset_info["num_classes"]
CLASS_NAMES = dataset_info["class_names"]

print(f"Dataset: Palm Ripeness Detection")
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Train/Val/Test: {dataset_info['n_train']}/{dataset_info['n_valid']}/{dataset_info['n_test']}")
print(f"\nGA best LR: {ga_history.get('best_lr', 'N/A')}")
print(f"GA best mAP: {ga_history.get('best_map', 'N/A')}")

## Step 3: Evaluation metrics table

In [ ]:
import pandas as pd

MODEL_LABELS = {
    "model1_baseline": "YOLOv4-tiny",
    "model2_es": "YOLOv4-tiny ES",
    "model3_ga": "YOLOv4-tiny (GA)",
    "model4_es_ga": "YOLOv4-tiny ES(GA)",
}

def build_table(split: str) -> pd.DataFrame:
    rows = []
    for r in metrics:
        m = r[split]
        rows.append({
            "Model": MODEL_LABELS.get(r["name"], r["name"]),
            "TP": m["TP"],
            "FP": m["FP"],
            "FN": m["FN"],
            "Precision": f"{m['Precision']:.4f}",
            "Recall": f"{m['Recall']:.4f}",
            "IoU": f"{m['IoU']*100:.2f}%",
            "F1": f"{m['F1']:.4f}",
            "mAP": f"{m['mAP']*100:.2f}%",
        })
    return pd.DataFrame(rows)

print("=" * 80)
print("VALIDATION SET")
print("=" * 80)
df_val = build_table("valid")
display(df_val)

print("\n" + "=" * 80)
print("TEST SET")
print("=" * 80)
df_test = build_table("test")
display(df_test)

df_val.to_csv("/content/metrics_validation.csv", index=False)
df_test.to_csv("/content/metrics_test.csv", index=False)
print("\nSaved CSVs to /content/")

## Step 4: Training loss curves

In [ ]:
import re
import matplotlib.pyplot as plt

_RE_ITER = re.compile(r"^\s*(\d+):\s*loss=")
_RE_LOSS = re.compile(r"avg\s+loss=(\d+\.\d+)")
_RE_MAP = re.compile(
    r"mean\s+average\s+precision\s*\(mAP@0\.50\)\s*=\s*([\d.]+)",
    re.IGNORECASE,
)
_RE_ANSI = re.compile(
    r"\x1b\[[0-9;]*[A-Za-z]"
    r"|\x1b\][^\x07\x1b]*(?:\x07|\x1b\\)"
    r"|[\x00-\x08\x0b-\x1f]"
)

def parse_log(log_path: Path):
    iters, losses, map_iters, maps = [], [], [], []
    last_iter = 0
    pending_map = False
    text = _RE_ANSI.sub("", log_path.read_text(errors="replace"))
    for line in text.splitlines():
        m_it = _RE_ITER.match(line)
        m_loss = _RE_LOSS.search(line)
        if m_it and m_loss:
            it = int(m_it.group(1))
            iters.append(it)
            losses.append(float(m_loss.group(1)))
            last_iter = it
            continue
        if "Calculating mAP" in line:
            pending_map = True
            continue
        m_map = _RE_MAP.search(line)
        if m_map and pending_map:
            val = float(m_map.group(1))
            map_iters.append(last_iter)
            maps.append(val / 100.0 if val > 1.0 else val)
            pending_map = False
    return iters, losses, map_iters, maps

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Training Loss Curves", fontsize=14, fontweight="bold")

for (key, label), ax in zip(MODEL_LABELS.items(), axes.flat):
    log_path = REPO / "logs" / f"{key}.log"
    if not log_path.exists():
        ax.set_title(f"{label} (log missing)")
        ax.axis("off")
        continue
    iters, losses, _, _ = parse_log(log_path)
    ax.plot(iters, losses, linewidth=0.7, color="#1976D2")
    ax.set_title(f"{label}  (n={len(iters)})")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Avg Loss")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/training_loss.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 5: mAP progression during training

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]

for (key, label), color in zip(MODEL_LABELS.items(), colors):
    log_path = REPO / "logs" / f"{key}.log"
    if not log_path.exists():
        continue
    _, _, map_iters, maps = parse_log(log_path)
    if maps:
        ax.plot(map_iters, [m*100 for m in maps], marker="o",
                label=label, color=color, linewidth=2)

ax.set_xlabel("Iteration")
ax.set_ylabel("mAP@0.50 (%)")
ax.set_title("mAP Progression During Training")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/content/map_progression.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 6: mAP comparison bar chart

In [ ]:
import numpy as np

model_labels = [MODEL_LABELS[r["name"]] for r in metrics]
val_maps = [r["valid"]["mAP"] * 100 for r in metrics]
test_maps = [r["test"]["mAP"] * 100 for r in metrics]

x = np.arange(len(model_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
b1 = ax.bar(x - width/2, val_maps, width, label="Validation", color="#1976D2")
b2 = ax.bar(x + width/2, test_maps, width, label="Test", color="#FF6F00")

for bar, val in list(zip(b1, val_maps)) + list(zip(b2, test_maps)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.2f}%", ha="center", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(model_labels, rotation=15)
ax.set_ylabel("mAP@0.50 (%)")
ax.set_title("Model Comparison: Validation vs Test")
ax.set_ylim(0, max(val_maps + test_maps) * 1.15)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("/content/map_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 7: GA optimization history

In [ ]:
if ga_history.get("history"):
    df_ga = pd.DataFrame(ga_history["history"])
    best_lr = ga_history["best_lr"]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for gen in sorted(df_ga["gen"].unique()):
        sub = df_ga[df_ga["gen"] == gen]
        ax1.scatter([gen] * len(sub), sub["mAP"], alpha=0.6, s=60)

    best_per_gen = df_ga.groupby("gen")["mAP"].max()
    ax1.plot(best_per_gen.index, best_per_gen.values, "r-o",
             linewidth=2, label="Best mAP per generation")
    ax1.set_xlabel("Generation")
    ax1.set_ylabel("mAP")
    ax1.set_title("GA: mAP Distribution per Generation")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    for gen in sorted(df_ga["gen"].unique()):
        sub = df_ga[df_ga["gen"] == gen]
        ax2.scatter(sub["lr"], sub["mAP"], label=f"Gen {gen}", s=60, alpha=0.7)
    ax2.axvline(x=best_lr, color="red", linestyle="--",
                label=f"Best LR={best_lr:.6f}")
    ax2.set_xlabel("Learning Rate")
    ax2.set_ylabel("mAP")
    ax2.set_title("GA: Learning Rate vs mAP")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("/content/ga_history.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nBest LR from GA: {best_lr:.6f}")
else:
    print("No GA history (skipped during training).")

## Step 8: Load models via OpenCV DNN

In [ ]:
import cv2
import numpy as np

def load_yolo_model(cfg_path: Path, weights_path: Path):
    net = cv2.dnn.readNetFromDarknet(str(cfg_path), str(weights_path))
    try:
        net.setPreferableBackend(cv2.dnn.DNN_BACKEND_CUDA)
        net.setPreferableTarget(cv2.dnn.DNN_TARGET_CUDA)
    except Exception:
        net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
    return net

BEST_KEY = "model3_ga"
best_cfg = REPO / "configs" / f"{BEST_KEY}.cfg"
best_weights = REPO / "weights" / f"{BEST_KEY}_best.weights"

best_net = load_yolo_model(best_cfg, best_weights)
print(f"Loaded: {BEST_KEY}")
print(f"Output layers: {best_net.getUnconnectedOutLayersNames()}")

In [ ]:
np.random.seed(42)
CLASS_COLORS = (np.random.uniform(0, 255, size=(NUM_CLASSES, 3))).astype(int).tolist()

def detect(net, img_bgr, conf_threshold=0.25, nms_threshold=0.4, input_size=416):
    h, w = img_bgr.shape[:2]
    blob = cv2.dnn.blobFromImage(img_bgr, 1/255.0, (input_size, input_size),
                                  swapRB=True, crop=False)
    net.setInput(blob)
    outputs = net.forward(net.getUnconnectedOutLayersNames())

    boxes, confidences, class_ids = [], [], []
    for out in outputs:
        for det in out:
            scores = det[5:]
            class_id = int(np.argmax(scores))
            confidence = float(scores[class_id])
            if confidence < conf_threshold:
                continue
            cx, cy, bw, bh = det[0]*w, det[1]*h, det[2]*w, det[3]*h
            x = int(cx - bw/2)
            y = int(cy - bh/2)
            boxes.append([x, y, int(bw), int(bh)])
            confidences.append(confidence)
            class_ids.append(class_id)

    if not boxes:
        return []

    idxs = cv2.dnn.NMSBoxes(boxes, confidences, conf_threshold, nms_threshold)
    if len(idxs) == 0:
        return []

    results = []
    for i in np.array(idxs).flatten():
        x, y, bw, bh = boxes[i]
        results.append((class_ids[i], confidences[i], (x, y, x+bw, y+bh)))
    return results

def draw_detections(img_bgr, detections):
    img = img_bgr.copy()
    for cid, conf, (x1, y1, x2, y2) in detections:
        color = CLASS_COLORS[cid]
        label = f"{CLASS_NAMES[cid]}: {conf:.2f}"
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(img, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
        cv2.putText(img, label, (x1 + 2, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    return img

## Step 9: Download test images from Roboflow

In [ ]:
import glob

if ROBOFLOW_API_KEY and ROBOFLOW_API_KEY != "YOUR_API_KEY":
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("tugas-akhir-pybma").project("palm-ripeness-detection")
    dataset = project.version(5).download("darknet")
    TEST_IMAGES = sorted(glob.glob(os.path.join(dataset.location, "test", "*.jpg")) +
                         glob.glob(os.path.join(dataset.location, "test", "*.png")))
    print(f"Got {len(TEST_IMAGES)} test images")
else:
    print("Skipping Roboflow download. Upload test images to /content/test_imgs/ manually")
    Path("/content/test_imgs").mkdir(exist_ok=True)
    TEST_IMAGES = sorted(glob.glob("/content/test_imgs/*.jpg") +
                         glob.glob("/content/test_imgs/*.png"))

## Step 10: Bounding box visualization on test images

In [ ]:
import random
random.seed(7)

sample = random.sample(TEST_IMAGES, min(6, len(TEST_IMAGES)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_path in zip(axes.flat, sample):
    img = cv2.imread(img_path)
    if img is None:
        ax.axis("off")
        continue
    detections = detect(best_net, img, conf_threshold=0.25)
    annotated = draw_detections(img, detections)
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    unique_cls = set(CLASS_NAMES[d[0]] for d in detections)
    ax.set_title(f"{Path(img_path).name[:30]} | {len(detections)} det | {unique_cls}",
                 fontsize=9)
    ax.axis("off")

plt.suptitle(f"Detection: {MODEL_LABELS[BEST_KEY]}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/detection_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 11: Side-by-side comparison of all 4 models

In [ ]:
all_nets = {}
for key, label in MODEL_LABELS.items():
    cfg = REPO / "configs" / f"{key}.cfg"
    w = REPO / "weights" / f"{key}_best.weights"
    if cfg.exists() and w.exists():
        all_nets[key] = (label, load_yolo_model(cfg, w))
        print(f"Loaded: {label}")

demo_img_path = random.choice(sample)
demo_img = cv2.imread(demo_img_path)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for (key, (label, net)), ax in zip(all_nets.items(), axes.flat):
    dets = detect(net, demo_img, conf_threshold=0.25)
    annotated = draw_detections(demo_img, dets)
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{label}  |  {len(dets)} detections")
    ax.axis("off")

plt.suptitle(f"Model Comparison: {Path(demo_img_path).name}",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/model_comparison_detection.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 12: Export all outputs

In [ ]:
import shutil

EXPORT_DIR = Path("/content/export_for_article")
EXPORT_DIR.mkdir(exist_ok=True)

for f in [
    "metrics_validation.csv", "metrics_test.csv",
    "training_loss.png", "map_progression.png",
    "map_comparison.png", "ga_history.png",
    "detection_samples.png", "model_comparison_detection.png",
]:
    src = Path(f"/content/{f}")
    if src.exists():
        shutil.copy(src, EXPORT_DIR / f)

summary_md = f'''# Palm Oil FFB Ripeness Detection - YOLOv4-tiny

## Setup
- Dataset: Roboflow tugas-akhir-pybma/palm-ripeness-detection v2
- Classes ({NUM_CLASSES}): {", ".join(CLASS_NAMES)}
- Distribution: Train={dataset_info["n_train"]}, Val={dataset_info["n_valid"]}, Test={dataset_info["n_test"]}
- GA Best Learning Rate: {ga_history.get("best_lr", "N/A")}
- Baseline Learning Rate: 0.00261

## Test Set Results

{df_test.to_markdown(index=False)}

## Validation Set Results

{df_val.to_markdown(index=False)}
'''
(EXPORT_DIR / "SUMMARY.md").write_text(summary_md)

shutil.make_archive("/content/palm_yolov4_results", "zip", EXPORT_DIR)
print(f"\nExport ready: /content/palm_yolov4_results.zip")
for f in sorted(EXPORT_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

In [ ]:
from google.colab import files
files.download("/content/palm_yolov4_results.zip")